# Dados e corpus de frases

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Igorbraziel/juristcu-clustering-tsdae/blob/main/notebooks/01_data_corpus.ipynb)

- **Objetivo:** descrever o JurisTCU e montar o corpus de frases sem rótulo para o TSDAE.
- **Entradas:** arquivo `doc.csv` do JurisTCU, no Hugging Face.
- **Saídas:** corpus de frases no Drive e tabelas descritivas em `results/tables/`.
- **Requisitos:** não requer GPU.

In [ ]:
# Configuração do ambiente. Fora do Colab, usa o ambiente local criado por `uv sync`.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # Trocar pelo nome de uma branch de trabalho para testá-la.
REPO_URL = "https://github.com/Igorbraziel/juristcu-clustering-tsdae.git"
REPO_DIR = "/content/juristcu-clustering-tsdae"


def run(command: str) -> None:
    """Executa um comando de shell e interrompe a célula em caso de erro."""
    subprocess.run(command, shell=True, check=True)


if IN_COLAB:
    # Clona o repositório, se necessário, e posiciona na versão remota da branch.
    run(f"test -d {REPO_DIR} || git clone -q {REPO_URL} {REPO_DIR}")
    run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
    run(f"git -C {REPO_DIR} checkout -q -B {BRANCH} FETCH_HEAD")
    # O uv costuma vir no Colab. Instala apenas se faltar.
    run("command -v uv > /dev/null || pip install -q uv")
    # --system: instala no Python do Colab, que já traz o torch com CUDA.
    # --no-sources: ignora o índice de torch CPU do pyproject.toml.
    # --no-config: ignora configurações locais do uv.
    # Instalação não editável: disponível sem reiniciar o ambiente de execução.
    # Nunca usar `uv sync` nem requirements.txt no Colab.
    run(f"uv pip install --system --no-config --no-sources -q {REPO_DIR}")
    # Tabelas e figuras são gravadas em results/, a partir da raiz do repositório.
    os.chdir(REPO_DIR)
    # Após atualizar o repositório com mudanças em src/, reiniciar o ambiente de execução.
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import hashlib
import importlib
import json

import matplotlib.pyplot as plt
import pandas as pd

# Atualiza o cache de importação para reconhecer o pacote recém-instalado.
importlib.invalidate_caches()

from juristcu import corpus, data, storage  # noqa: E402
from juristcu.config import (  # noqa: E402
    AREA_COLUMN,
    EXCERPT_COLUMN,
    MIN_TOPIC_SIZE,
    N_SENTENCES,
    SEED,
    TEXT_COLUMN,
    TOPIC_COLUMN,
)

storage.save_environment("01")

## 1. Carregamento

- Carrega o arquivo `doc.csv` diretamente. O visualizador do Hugging Face falha com este conjunto.
- São esperados 16.045 documentos.

In [ ]:
documents = data.load_documents()
documents.shape

In [ ]:
documents.head()

## 2. Análise exploratória

- Distribuição das áreas e dos temas, para avaliar o equilíbrio entre eles.
- O número de áreas define o k do agrupamento principal.
- Os temas com ao menos 20 documentos definem o k da análise por tema.
- Tamanho dos enunciados e dos excertos em palavras, com percentis.
- O percentil 99 em palavras orienta o limite de tokens na codificação (`ENCODE_MAX_SEQ_LENGTH`).
- Presença de marcação HTML e de quebras de linha escritas como texto.

In [ ]:
area_distribution = data.label_distribution(documents, AREA_COLUMN)
area_distribution

In [ ]:
topic_distribution = data.label_distribution(documents, TOPIC_COLUMN)
topic_distribution

In [ ]:
balance = pd.DataFrame(
    {
        AREA_COLUMN: data.summarize_distribution(area_distribution),
        TOPIC_COLUMN: data.summarize_distribution(topic_distribution),
    }
)
balance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.35 * len(area_distribution) + 1))
ax.barh(area_distribution[AREA_COLUMN].astype(str), area_distribution["count"])
ax.invert_yaxis()
ax.set_xlabel("Documentos")
ax.set_title("Documentos por área")
fig.tight_layout()

In [ ]:
# Temas distintos por área e temas com ao menos MIN_TOPIC_SIZE documentos.
topic_sizes = documents.groupby([AREA_COLUMN, TOPIC_COLUMN]).size()
topics_per_area = pd.DataFrame(
    {
        "topics": topic_sizes.groupby(level=0).size(),
        f"topics_min_{MIN_TOPIC_SIZE}": (topic_sizes >= MIN_TOPIC_SIZE).groupby(level=0).sum(),
    }
).sort_values("topics", ascending=False)
topics_per_area

In [ ]:
# Um mesmo tema em mais de uma área indica que TEMA não é aninhado em AREA.
areas_per_topic = documents.groupby(TOPIC_COLUMN)[AREA_COLUMN].nunique()
(areas_per_topic > 1).sum()

In [ ]:
word_counts = pd.DataFrame(
    {
        column: documents[column].fillna("").str.split().str.len()
        for column in (TEXT_COLUMN, EXCERPT_COLUMN)
    }
)
length_percentiles = word_counts.describe(percentiles=[0.5, 0.9, 0.95, 0.99])
length_percentiles

In [ ]:
# Marcação a remover antes da segmentação; informa também a limpeza do ENUNCIADO.
markup = pd.DataFrame(
    {
        column: {
            "html": documents[column].str.contains(r"<[^>]+>", na=False).mean(),
            "quebra_escrita": documents[column].str.contains(r"\n", regex=False, na=False).mean(),
            "quebra_real": documents[column].str.contains("\n", regex=False, na=False).mean(),
            "ausente": documents[column].isna().mean(),
        }
        for column in (TEXT_COLUMN, EXCERPT_COLUMN)
    }
)
markup

In [ ]:
# Exemplos de enunciados com marcação, antes e depois da limpeza.
with_markup = documents.loc[documents[TEXT_COLUMN].str.contains(r"<[^>]+>", na=False), TEXT_COLUMN]
pd.DataFrame({"original": with_markup.head(3), "limpo": with_markup.head(3).map(data.clean_text)})

## 3. Corpus de frases sem rótulo

- Fonte: ENUNCIADO e EXCERTO, sem áreas, temas ou outros metadados.
- Limpeza: remove marcação HTML e quebras de linha (`data.clean_text`).
- Segmentação com o Punkt do NLTK e abreviações jurídicas. Frases de 5 a 80 palavras, sem duplicatas.
- Amostra de 20 mil frases com semente fixa, a mesma para as 3 sementes do TSDAE.
- A segmentação dos 16 mil documentos leva alguns minutos.

In [ ]:
corpus.ensure_nltk_resources()
funnel = corpus.corpus_funnel(documents)
funnel

In [ ]:
sentences = corpus.build_corpus(documents, n_sentences=N_SENTENCES, seed=SEED)
len(sentences)

## 4. Inspeção manual

- Amostra aleatória de frases, para detectar ruído de segmentação.
- Tamanho das frases em palavras, para conferir o filtro de 5 a 80 palavras.
- Frases mais curtas e mais longas da amostra, onde os erros de segmentação se concentram.

In [ ]:
pd.Series(sentences).sample(n=20, random_state=SEED).to_list()

In [ ]:
pd.Series(sentences).str.split().str.len().describe()

In [ ]:
# Frases curtas e longas concentram erros de segmentação.
by_length = pd.Series(sentences, name="sentence").to_frame()
by_length["words"] = by_length["sentence"].str.split().str.len()
pd.concat([by_length.nsmallest(10, "words"), by_length.nlargest(5, "words")])

## 5. Salvamento

- Corpus de frases em `sentences.json`, no diretório de artefatos (Drive no Colab).
- O SHA-256 do corpus permite conferir que todos usam a mesma amostra.
- Tabelas descritivas e figura da distribuição das áreas em `results/`.

In [ ]:
sentences_path = storage.artifacts_dir() / "sentences.json"
payload = json.dumps(sentences, ensure_ascii=False)
sentences_path.write_text(payload, encoding="utf-8")
corpus_sha256 = hashlib.sha256(payload.encode("utf-8")).hexdigest()
sentences_path, corpus_sha256

In [ ]:
storage.save_table(area_distribution, "area_distribution")
storage.save_table(topic_distribution, "topic_distribution")
storage.save_table(balance.rename_axis("statistic").reset_index(), "label_balance")
storage.save_table(topics_per_area.reset_index(), "topics_per_area")
storage.save_table(length_percentiles.rename_axis("statistic").reset_index(), "text_length")
storage.save_table(markup.rename_axis("check").reset_index(), "text_markup")
storage.save_table(funnel.assign(sha256=corpus_sha256), "corpus_funnel")
storage.save_figure(fig, "area_distribution")